# 06 · Saying "no sign here": empty scenes, decoys and the threshold τ

Every scene up to v2 held exactly one sign, so those models never had to say "there's nothing here". Shown
an empty street, they still point somewhere and name a class. v3a and v3a.1 add that missing answer:

* **presence** = the heatmap's single brightest cell, as a probability (`max sigmoid`). High means "a sign is
  here".
* **τ (tau)** = a cutoff on presence. Above τ the model says "sign"; below, "no sign". τ is chosen on
  **val_empty** (5,000 scenes with no sign) so that only 1% of them cross it: a 1% **false-alarm** rate.
* **found** = presence ≥ τ **and** peak in the box **and** class right. It's joint accuracy that also has to
  clear τ.

The two versions differ in what the model saw during training:

| version | run name | per epoch of 20,000 training scenes |
|---|---|---|
| v3a | `peak_empty` | 16,000 sign scenes + 4,000 **empty** scenes (background only) |
| v3a.1 | `peak_decoy` | 16,000 sign scenes + 4,000 **decoys**: a sign scene with the sign's pixels replaced by a patch of another background, same size and spot |

Decoys exist because v3a learned a shortcut: "a pasted rectangle with sharp edges means sign". Shown a pasted
patch with no sign in it, it still said "sign" between 1 time in 11 and 1 time in 4, depending on the seed.

**You'll see:** each kind of no-sign scene, live presence scores and τ, the v3a → v3a.1 change across three
seeds, and the real-photo results, including the remaining gap between real and pasted signs.

**Needs:** the v3a files (`python scripts/make_empty_dataset.py`) and trained `peak_decoy` / `peak_empty`
runs. **Writes:** nothing, unless you turn on a flag.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_decoy"  # the model to score live (v3a.1 seed 0); try "peak_empty" for v3a
BEFORE = "peak_empty"  # the model RUN_NAME is compared against in the "run it for real" cells
REBUILD = False  # True: rebuild the no-sign data files with scripts/make_empty_dataset.py (~3 min; byte-identical)
RUN_FOR_REAL = False  # True: record presence on val and real_dev, the seam report and a comparison (~1 min)
SCORE_TEST = False  # True: score test and real_test ONCE for RUN_NAME (refused if already scored)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing; then find both runs.
HOW = "python scripts/make_empty_dataset.py --config configs/peak_fresh_aug.yaml"
for stem in ("val_empty", "val_decoy_plain", "val_decoy_aug", "real_empty_dev"):
    require(f"data/processed/{stem}.npz", how=HOW)
require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")
RUN_DIR, BEFORE_DIR = find_run(RUN_NAME), find_run(BEFORE)
CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)  # relative, as the scripts record it in their JSON
EVAL = ROOT / "outputs/eval"
print("run:   ", RUN_DIR.name)
print("before:", BEFORE_DIR.name)

## 1. The kinds of no-sign scenes

Four files of scenes with no sign, used only for scoring (never for training):

* **val_empty**: a val COCO background, cropped and resized exactly like a sign scene, with nothing pasted.
  τ is chosen on these.
* **val_decoy_plain**: a val background with a patch of *another* background pasted where a sign would go,
  through the same paste code val signs use. It tests "does a pasted rectangle alone fool the model?"
* **val_decoy_aug**: the same 5,000 draws, pasted through the *training* code path (rotation, perspective,
  contrast). Same background, same patch, same spot as plain; only the paste differs.
* **real_empty_dev**: 128×128 windows cut from real GTSDB photos where no labeled sign is. Half are anywhere
  in the photo (**uniform**), half at the height where that photo's signs are (**band**), which is harder,
  since that's where poles, windows and other signs live.

`EmptyScenes` serves these files in the same format as sign scenes, with class `-100` (the "ignore" class, so
no class loss is computed) and an all-zero target heatmap.

In [ ]:
# Show the first 6 scenes of each no-sign file, one row per file.
import matplotlib.pyplot as plt

from findobject.data.dataset import EmptyScenes

NEG_FILES = ("val_empty", "val_decoy_plain", "val_decoy_aug", "real_empty_dev")
neg = {stem: EmptyScenes(ROOT / "data/processed" / f"{stem}.npz") for stem in NEG_FILES}
fig, axes = plt.subplots(4, 6, figsize=(13, 9))
for row, stem in zip(axes, NEG_FILES):
    for i, ax in enumerate(row):
        ax.imshow(neg[stem][i].image.permute(1, 2, 0))  # [3,128,128] -> [128,128,3] for imshow
        ax.set_axis_off()
    row[0].set_title(f"{stem} ({len(neg[stem]):,} scenes)", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

Compare the plain and aug rows column by column: the same patch on the same background, pasted two ways. The
patches are hard to spot by eye, which is the point. To the model the giveaway was never the content of the
patch, but its sharp pasted outline.

### Training decoys

The decoys v3a.1 *trains* on are made on the fly, like all its training scenes (notebook 01's "fresh" scenes).
Scenes 16,000–19,999 of every epoch are decoys. Decoy *i* is built from the same random draws as sign scene
*i* would be (same background, same crop, same box), then the sign is swapped for a background patch. So we
can rebuild sign scene *i* with decoys turned off and put the two side by side.

In [ ]:
# Scene 16,000 + k of epoch 1, built twice: as v3a.1 trains on it (a decoy), and with decoys off (the sign scene).
import copy

import yaml
from matplotlib.patches import Rectangle

from findobject.training.train_data import epoch_datasets

raw = yaml.safe_load((ROOT / "configs/peak_decoy.yaml").read_text())
no_decoy = copy.deepcopy(raw["data"])
no_decoy["n_train_decoy"] = 0  # same seed, same epoch: scene i draws the same background, crop and box
decoys = epoch_datasets(raw["data"], raw["train"]["seed"])(1)
signs = epoch_datasets(no_decoy, raw["train"]["seed"])(1)

fig, axes = plt.subplots(2, 6, figsize=(13, 4.8))
for k in range(6):
    i = 16000 + k
    s, d = signs[i], decoys[i]
    x1, y1, x2, y2 = s.box.tolist()
    for ax, scene, label in ((axes[0, k], s, f"sign scene {i}"), (axes[1, k], d, f"decoy {i}, class {int(d.class_id)}")):
        ax.imshow(scene.image.permute(1, 2, 0))
        ax.add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=0.8))
        ax.set_title(label, fontsize=8), ax.set_axis_off()
plt.tight_layout()
plt.show()

The green box is drawn on both rows only to show where the sign was. The decoy's target is "no sign
anywhere": class -100 and an all-zero heatmap, just like an empty scene.

## 2. Presence and τ, live

`score_scenes` runs the model over a file and keeps, for each scene, its presence, predicted class, and
whether the peak landed in the box. `threshold_at_false_alarm(neg, 0.01)` then picks τ: sort the 5,000
val_empty presences from high to low, and set τ just above the 51st, so that at most 50 (1%) are at or
above it.

In [ ]:
# Score val (5,000 sign scenes) and the four no-sign files with the trained model (~30 s on MPS).
import numpy as np
from torch.utils.data import DataLoader

from findobject.data.dataset import SignScenes
from findobject.eval.presence import threshold_at_false_alarm
from findobject.eval.score import score_scenes
from findobject.training.checkpoint import load_model
from findobject.utils.device import pick_device

device = pick_device("auto")
model = load_model(CKPT, device)
scores = {stem: score_scenes(model, DataLoader(ds, batch_size=256), device) for stem, ds in
          [("val", SignScenes(ROOT / "data/processed/val.npz"))] + list(neg.items())}
pos = scores["val"]
for stem, s in scores.items():
    print(f"{stem:16} {len(s.presence):5} scenes, median presence {np.median(s.presence):.3f}")

In [ ]:
# Choose τ on val_empty, then count: found among sign scenes, false alarms among each no-sign file.
stored = read_json(EVAL / RUN_DIR.name / "presence_val.json")
tau = threshold_at_false_alarm(scores["val_empty"].presence, 0.01)
print(f"τ = {tau:.6f}   (stored in presence_val.json: {stored['tau']:.6f})")

found = (pos.presence >= tau) & (pos.y_pred == pos.y_true) & pos.hit
print(f"found       {found.mean():.4f}   (stored {stored['report']['all']['found']['value']:.4f})")
print(f"joint       {((pos.y_pred == pos.y_true) & pos.hit).mean():.4f}   (ignores τ)")
for stem in NEG_FILES:
    fa = (scores[stem].presence >= tau).mean()
    note = {"val_empty": "1% by construction", "real_empty_dev": "real windows; see section 4"}.get(stem, "")
    if stem in stored["report"]["decoy_false_alarm"]:
        note = f"stored {stored['report']['decoy_false_alarm'][stem]['value']:.4f}"
    print(f"false alarm {fa:.4f}   {stem:16} {note}")

These should match the stored numbers exactly, or to the last digit at most (the Mac GPU doesn't always add in
the same order). The histogram shows *why* one cutoff can separate the groups: sign scenes pile up near 1,
empties near 0, and the decoys sit in between, mostly below τ.

In [ ]:
# Histogram of presence for each group, with τ as a dashed line. Log y-axis: the tails are small but matter.
bins = np.linspace(0, 1, 51)
plt.figure(figsize=(10, 4))
for stem, color in (("val", "tab:green"), ("val_empty", "tab:gray"), ("val_decoy_aug", "tab:red"),
                    ("real_empty_dev", "tab:blue")):
    plt.hist(scores[stem].presence, bins=bins, histtype="step", linewidth=1.5, color=color, label=stem)
plt.axvline(tau, color="black", linestyle="--", label=f"τ = {tau:.3f}")
plt.yscale("log"), plt.xlabel("presence (max heatmap probability)"), plt.ylabel("scenes")
plt.title(f"{RUN_DIR.name}: presence by kind of scene"), plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

In [ ]:
# The 8 real no-sign windows the model is most sure contain a sign. Red title = above τ (a false alarm).
real_neg = scores["real_empty_dev"].presence
stratum = np.load(ROOT / "data/processed/real_empty_dev.npz")["stratum"]  # 0 = uniform, 1 = band
top = np.argsort(real_neg)[::-1][:8]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for ax, i in zip(axes, top):
    ax.imshow(neg["real_empty_dev"][int(i)].image.permute(1, 2, 0)), ax.set_axis_off()
    ax.set_title(f"{real_neg[i]:.2f} ({'band' if stratum[i] else 'uniform'})", fontsize=9,
                 color="red" if real_neg[i] >= tau else "black")
plt.tight_layout()
plt.show()

For the default run, the windows that fool it most include real signs that aren't among the 43 classes (a
blue crosswalk sign, a no-stopping sign: GTSDB labels only its own classes, so these count as "no sign"), car
fronts, and big lettering. Most sit in the band stratum, at sign height.

## 3. v3a → v3a.1 across three seeds (read from disk)

One seed can be lucky, so each version was trained three times. The table reads each run's stored
`presence_val.json`. The question decoys were meant to answer is in the last two columns.

In [ ]:
# One row per seed of each version: τ, found and joint on val, and false alarms on the two decoy files.
print(f"{'run':30} {'τ':>6} {'found':>7} {'joint':>7} {'AUROC':>7}  {'decoy FA plain':>14} {'decoy FA aug':>12}")
for name in ("peak_empty", "peak_empty_s1", "peak_empty_s2", "peak_decoy", "peak_decoy_s1", "peak_decoy_s2"):
    r = read_json(EVAL / find_run(name).name / "presence_val.json")
    a, d = r["report"]["all"], r["report"]["decoy_false_alarm"]
    print(f"{find_run(name).name:30} {r['tau']:6.3f} {a['found']['value']:7.4f} {a['joint']['value']:7.4f} "
          f"{a['auroc']['value']:7.4f}  {d['val_decoy_plain']['value']:14.4f} {d['val_decoy_aug']['value']:12.4f}")

Decoy false alarms fall on every seed, from 9–25% to 1.4–2.2%: five to ten times fewer. They also stop
swinging between seeds, which v3a's did (9% to 19% on plain decoys). That's the change v3a.1 was built for.
The cost is small: found and joint on val move by less than half a point, and not always down.

**AUROC** (area under the ROC curve) asks: pick one sign scene and one empty scene at random; how often does
the sign scene get the higher presence? 1.0 is perfect separation, 0.5 is a coin flip. It doesn't depend on
τ at all.

## 4. Real photos (read from disk)

`evaluate_presence.py --split real_dev` scores the 645 real sign windows against 1,200 real no-sign windows,
using the τ chosen on synthetic val_empty. Intervals here come from resampling whole *photos*, since windows
cut from one photo aren't independent.

It also scores the **hybrid** twin of each real window (the same real sign cut out and pasted on a COCO
background, notebook 05). If real and hybrid scored the same, the model would treat real signs and pasted
signs alike. They don't: the **seam** columns show how many real signs fall below τ although their pasted
twin clears it.

In [ ]:
# Real-photo results for each seed: found among real signs (and among signs never seen in training), false
# alarms on real no-sign windows by stratum, and how often real and pasted twins clear τ ("said yes").
print(f"{'run':30} {'found':>6} {'unseen':>7}  {'FA uniform':>10} {'FA band':>8}  "
      f"{'real yes':>8} {'hybrid yes':>10} {'only real':>9} {'only hybrid':>11}")
for name in ("peak_empty", "peak_empty_s1", "peak_empty_s2", "peak_decoy", "peak_decoy_s1", "peak_decoy_s2"):
    r = read_json(EVAL / find_run(name).name / "presence_real_dev.json")["report"]
    sub, st, seam = r["subsets"], r["strata"], r["seam"]
    print(f"{find_run(name).name:30} {sub['all']['found']['value']:6.3f} {sub['unseen']['found']['value']:7.3f}  "
          f"{st['uniform']['false_alarm']['value']:10.4f} {st['band']['false_alarm']['value']:8.4f}  "
          f"{seam['real_yes']['value']:8.3f} {seam['hybrid_yes']['value']:10.3f} {seam['only_real']:9d} {seam['only_hybrid']:11d}")

* **found** is lower than on synthetic val (0.90–0.96 vs about 0.98): real photos are harder.
* **FA band** is higher than **FA uniform** in every run: windows at sign height hold more sign-like things.
* **only hybrid** counts real windows that fall below τ while their pasted twin clears it. **only real** is
  the reverse, and it's 0 or 1 in every run. So the model is more confident about pasted signs than real ones.
  Decoys didn't shrink that gap: only hybrid is 21–56 across v3a's seeds and 25–36 across v3a.1's, which is
  within seed-to-seed noise. Decoys fixed the outline shortcut on *no-sign* scenes, not this gap on sign
  scenes.

The **seam report** breaks that gap down by whether the pasted twin ended up bigger, about the same, or
smaller than the real sign:

In [ ]:
# Median (hybrid presence - real presence) by size group, with its 95% interval, for RUN_NAME.
path = EVAL / RUN_DIR.name / "seam_real_dev.json"
if not path.exists():
    print(f"{path.relative_to(ROOT)} doesn't exist yet: turn on RUN_FOR_REAL below to make it.")
else:
    seam = read_json(path)
    print(f"sizes count as 'similar' within {seam['size_match_px']} px")
    for group, g in seam["median_gap"].items():
        print(f"{group:15} n={g['n']:3}   median gap {g['value']:+.3f}  [{g['lo']:+.3f}, {g['hi']:+.3f}]")
    s = seam["share_hybrid_higher"]
    print(f"hybrid scores higher than its real twin in {s['value']:.1%} of pairs")

The gap is positive in every group, even when the pasted twin is *bigger*. So it isn't only that pasted signs
come out larger. Something about a real sign in its real surroundings lowers the model's confidence.

## Run it for real

* **`REBUILD`**: rebuilds the seven no-sign files. Every scene has its own fixed random seed, so the files come
  out byte-identical, and `empty_manifest.json` records their SHA-256 fingerprints to check against.
* **`RUN_FOR_REAL`**, in order (each step reads the one before):
  1. `evaluate_presence --split val` chooses τ and writes `presence_val.json`.
  2. `--split real_dev` reads that τ, and refuses if it belongs to a different checkpoint.
  3. `seam_report` reads the real_dev result and writes `seam_real_dev.json`.
  4. `compare_presence` compares RUN_NAME with BEFORE on the same real_dev windows. It refuses if the two
     results were scored on different data files.
* **`SCORE_TEST`**: test and real_test, once per run. Every run above has already been scored, so these exit
  with code 2 and the cell raises an error. That's the test guard working, not a bug.

In [ ]:
# Rebuild the no-sign data files. Only runs when REBUILD = True.
run_script("make_empty_dataset", "--config", "configs/peak_fresh_aug.yaml", enabled=REBUILD)

In [ ]:
# Record presence on val (chooses τ), then on real_dev (uses it). Only runs when RUN_FOR_REAL = True.
run_script("evaluate_presence", "--checkpoint", CKPT, "--split", "val", enabled=RUN_FOR_REAL)
run_script("evaluate_presence", "--checkpoint", CKPT, "--split", "real_dev", enabled=RUN_FOR_REAL)

In [ ]:
# The seam report, and RUN_NAME vs BEFORE on the same real windows. Only runs when RUN_FOR_REAL = True.
run_script("seam_report", "--run", RUN_DIR.name, enabled=RUN_FOR_REAL)
run_script("compare_presence", "--a", RUN_DIR.name, "--b", BEFORE_DIR.name, "--split", "real_dev", enabled=RUN_FOR_REAL)

In [ ]:
# Score test and real_test (once per run). Only runs when SCORE_TEST = True.
run_script("evaluate_presence", "--checkpoint", CKPT, "--split", "test", enabled=SCORE_TEST)
run_script("evaluate_presence", "--checkpoint", CKPT, "--split", "real_test", enabled=SCORE_TEST)

**Next:** `07_inference_demo.ipynb`: run the model on any image, including your own.